# 04 — Clustering (Unsupervised Learning)

## What is Clustering?

**Clustering** is an **unsupervised** learning technique that groups data points so that points within the same cluster are more similar to each other than to points in other clusters — **without any labelled training data**.

### Algorithms Covered
1. **K-Means** — partition into k clusters via centroid iteration
2. **DBSCAN** — density-based clustering; discovers clusters of arbitrary shape

### Use Cases
- Customer segmentation
- Document grouping
- Anomaly detection
- Image compression

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

print('Libraries imported successfully!')

## 2. K-Means Clustering

### How K-Means Works
1. Randomly initialise **k** centroids
2. Assign each point to the nearest centroid
3. Recompute centroids as the mean of their assigned points
4. Repeat steps 2–3 until convergence

In [ ]:
# Generate synthetic blobs
X_blobs, y_true = make_blobs(
    n_samples=300, centers=4, cluster_std=0.8, random_state=42
)

plt.figure(figsize=(6, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap='Set1', alpha=0.7)
plt.title('Generated Blobs (True Labels)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.show()

### 2.1 The Elbow Method — Choosing k

In [ ]:
inertia = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, 'bo-', markersize=6)
plt.axvline(4, color='red', linestyle='--', label='Optimal k=4')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-cluster Sum of Squares)')
plt.title('Elbow Method')
plt.legend()
plt.tight_layout()
plt.show()

### 2.2 Fit K-Means with k=4

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_blobs)
centers   = kmeans.cluster_centers_

sil_score = silhouette_score(X_blobs, labels_km)
print(f'Silhouette Score: {sil_score:.4f}  (closer to 1 is better)')

plt.figure(figsize=(6, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_km, cmap='Set1', alpha=0.7)
plt.scatter(centers[:, 0], centers[:, 1],
            marker='X', s=200, c='black', zorder=5, label='Centroids')
plt.title('K-Means Clustering (k=4)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.tight_layout()
plt.show()

## 3. DBSCAN — Density-Based Clustering

K-Means struggles with non-spherical clusters. **DBSCAN** handles arbitrarily shaped clusters and can identify outliers.

### Key Parameters
- **`eps`** — maximum neighbourhood radius
- **`min_samples`** — minimum points to form a dense region (core point)

In [ ]:
# Non-linear 'moons' dataset where K-Means fails
X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

# K-Means on moons (fails)
labels_km_moons = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons)

# DBSCAN on moons (succeeds)
db = DBSCAN(eps=0.2, min_samples=5)
labels_db = db.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km_moons, cmap='Set1', alpha=0.7)
axes[0].set_title('K-Means on Moons (Fails)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db, cmap='Set1', alpha=0.7)
axes[1].set_title('DBSCAN on Moons (Succeeds)')

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print(f'DBSCAN found {len(set(labels_db)) - (1 if -1 in labels_db else 0)} cluster(s)')
print(f'Noise points: {np.sum(labels_db == -1)}')

## 4. Real-World Example — Customer Segmentation

In [ ]:
np.random.seed(0)
n = 200

customers = pd.DataFrame({
    'AnnualIncome_k': np.concatenate([
        np.random.normal(30, 5, n // 4),
        np.random.normal(60, 8, n // 4),
        np.random.normal(90, 7, n // 4),
        np.random.normal(45, 6, n // 4),
    ]),
    'SpendingScore': np.concatenate([
        np.random.normal(20, 5, n // 4),
        np.random.normal(70, 8, n // 4),
        np.random.normal(50, 6, n // 4),
        np.random.normal(85, 5, n // 4),
    ])
})

# Scale and cluster
scaler = StandardScaler()
X_cust = scaler.fit_transform(customers)

km_cust = KMeans(n_clusters=4, random_state=42, n_init=10)
customers['Segment'] = km_cust.fit_predict(X_cust)

seg_labels = {0: 'Budget Shoppers', 1: 'High Earners / High Spenders',
              2: 'Mid-Income / Mid-Spend', 3: 'Low Income / High Spend'}
customers['Segment_Label'] = customers['Segment'].map(seg_labels)

plt.figure(figsize=(9, 6))
for seg, grp in customers.groupby('Segment_Label'):
    plt.scatter(grp['AnnualIncome_k'], grp['SpendingScore'],
                label=seg, alpha=0.7, s=60)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1–100)')
plt.title('Customer Segmentation via K-Means')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(customers.groupby('Segment_Label')[['AnnualIncome_k', 'SpendingScore']].mean().round(1))

## 5. K-Means vs DBSCAN — When to Use Which?

| | K-Means | DBSCAN |
|---|---------|--------|
| **Shape** | Spherical / convex clusters | Arbitrary shapes |
| **Outliers** | Sensitive | Marks outliers as noise (-1) |
| **k required?** | Yes | No (auto-detects) |
| **Scalability** | Fast on large data | Slower for large data |

## Summary

✅ Applied K-Means clustering and used the Elbow Method to choose k  
✅ Evaluated cluster quality with the Silhouette Score  
✅ Showed DBSCAN's advantage on non-spherical data  
✅ Applied clustering to a customer segmentation problem  

---

🎉 **Congratulations!** You have completed the Machine Learning Basics Hands-on series.

| Notebook | Topic |
|----------|-------|
| 01 | Introduction & Data Preprocessing |
| 02 | Linear Regression |
| 03 | Classification (Logistic Regression, Decision Tree, Random Forest) |
| 04 | Clustering (K-Means, DBSCAN) |